# NRLMF Simulation — Local Notebook Runner

Open multiple copies of this notebook in VS Code and set different `DATASET_IDS` in each to run in parallel.

**Each notebook is responsible for a slice of `dataset_grid`** (i.e. specific `feature_id × repeat_id` combos).  
All train sizes and hyperparameter combos for those datasets are run automatically.

Results are saved to a per-notebook file: `results_nrlmf_local_{NOTEBOOK_ID}.gz`

Unlike DRIMC, NRLMF is pure Python (no R/rpy2 required).

## Cell 1 — Configure which datasets this notebook runs

In [10]:
# ============================================================
# USER SETTINGS — change these per notebook instance
# ============================================================

# A unique label for this notebook — used in the output filename
# e.g. 'nb0', 'nb1', 'nb2' ... open one notebook per label
NOTEBOOK_ID = "nb0"

# Which dataset_grid indices (i_ds) this notebook will process.
# dataset_grid has 100 entries (10 feature sizes x 10 repeats),
# indexed 0..99. Split them across notebooks however you like.
#
# Example splits across 4 notebooks:
#   nb0: list(range(0,  25))   # feature_id 0-1, all repeats
#   nb1: list(range(25, 50))   # feature_id 2-3, all repeats
#   nb2: list(range(50, 75))   # feature_id 4-6, all repeats
#   nb3: list(range(75, 100))  # feature_id 7-9, all repeats

DATASET_IDS = list(range(0, 10))  # <-- change this per notebook

BASE_SEED = 123456  # must match HPC runs for reproducible splits

# ============================================================

## Cell 2 — Paths

In [ ]:
import os
import sys
import gzip
import pickle
import warnings

import numpy as np
from tqdm import TqdmSynchronisationWarning, tqdm

warnings.simplefilter("ignore", TqdmSynchronisationWarning)

# ====== user paths — adjust to your local machine ======
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"  # <-- change to local root
PATH_DATA = os.path.join(PATH_ROOT, "datasets/simulations/n_features")
PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/simulations/n_features")
PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")

for p in (PATH_OUTPUT, PATH_ARCHIVE):
    os.makedirs(p, exist_ok=True)

sys.path.append(PATH_ROOT)

print(f"Output will be saved to: {PATH_OUTPUT}")
print(f"Notebook ID: {NOTEBOOK_ID}  |  Datasets to run: {len(DATASET_IDS)}")

Output will be saved to: /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features
Notebook ID: nb0  |  Datasets to run: 10


## Cell 3 — Imports

In [12]:
from sgimc.utils import mc_split, get_submatrix
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import ParameterGrid, ShuffleSplit, train_test_split
from scipy.special import expit

from PyDTI3.nrlmf import NRLMF

print("Imports OK.")

Imports OK.


## Cell 4 — Helper functions

In [13]:
def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]
    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN
    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])
    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])
    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)
    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]


def build_sim_matrices(U, V):
    """
    Compute Jaccard similarity matrices from U and V.
    Cached per dataset — only recomputed when the dataset changes.
    Returns (simD, simT) as numpy arrays.
    """
    simD = 1 - pairwise_distances(U, metric="jaccard")
    simT = 1 - pairwise_distances(V, metric="jaccard")
    return simD, simT


print("Helpers defined.")

Helpers defined.


## Cell 5 — Build grids

In [14]:
n_features_grid = np.arange(50, 501, 50)
n_repeats = 10
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for feature_id, n_features in enumerate(n_features_grid):
    for repeat_id in range(n_repeats):
        dataset_grid.append(
            {
                "feature_id": int(feature_id),
                "n_features": int(n_features),
                "repeat_id": int(repeat_id),
                "filename": os.path.join(
                    PATH_DATA, filename_template.format(n_features, repeat_id)
                ),
            }
        )

grid_dataset = ParameterGrid(
    {
        "train_size": np.arange(0.05, 0.51, 0.05),
        "n_splits": [3],
        "val_size": [0.20],
    }
)

grid_model = ParameterGrid(
    {
        "c": [1, 5, 10],  # confidence level — main tuning parameter
        "K1": [5],  # neighbourhood size for drugs (fixed)
        "K2": [5],  # neighbourhood size for targets (fixed)
        "r": [25],  # rank (fixed, matches other methods)
        "lambda_d": [0.125],  # drug regularization (fixed)
        "lambda_t": [0.125],  # target regularization (fixed)
        "alpha": [0.25],  # fixed
        "beta": [0.125],  # fixed
        "theta": [0.5],  # fixed
        "max_iter": [100],  # fixed
    }
)

# only the datasets assigned to this notebook
my_datasets = [dataset_grid[i] for i in DATASET_IDS]

n_dt = len(list(grid_dataset))
n_m = len(list(grid_model))
total = len(my_datasets) * n_dt * n_m
print(f"Datasets assigned : {len(my_datasets)}")
print(f"Train size levels : {n_dt}")
print(f"Hyperparam combos : {n_m}")
print(f"Total combos      : {total}")

Datasets assigned : 10
Train size levels : 10
Hyperparam combos : 3
Total combos      : 300


## Cell 6 — Run

Progress bars are shown per dataset.  
Results are **incrementally saved** after each dataset finishes — if the kernel dies mid-run you keep completed datasets.

In [15]:
all_results = []
_sim_cache = {}  # cache similarity matrices per (feature_id, repeat_id)

outfile = os.path.join(PATH_OUTPUT, f"results_nrlmf_local_{NOTEBOOK_ID}.gz")

for i_ds, ds in enumerate(tqdm(my_datasets, desc="Datasets")):

    # ---- load dataset ----
    with gzip.open(ds["filename"], "rb") as fin:
        data = pickle.load(fin)

    U = data["X"]
    V = data["Y"]
    Y = data["R_noisy"].astype(float)
    Y_true = data["R"]

    # ---- similarity matrices (cached per dataset) ----
    cache_key = (ds["feature_id"], ds["repeat_id"])
    if cache_key not in _sim_cache:
        print(
            f"  Building sim matrices for feature_id={ds['feature_id']}, repeat_id={ds['repeat_id']}"
        )
        _sim_cache[cache_key] = build_sim_matrices(U, V)
    simD, simT = _sim_cache[cache_key]

    dataset_results = []

    for i_dt, par_dtst in enumerate(grid_dataset):

        # split RNG — same formula as HPC scripts
        split_seed = BASE_SEED + ds["feature_id"] * 1000 + ds["repeat_id"] * 100 + i_dt
        rng_split = np.random.RandomState(split_seed)

        # dev/test split
        dvlp_size, test_size = 0.9, 0.1
        ind_dvlp, ind_test = next(
            mc_split(
                Y,
                n_splits=1,
                random_state=rng_split,
                train_size=dvlp_size,
                test_size=test_size,
            )
        )
        Y_test = get_submatrix(Y_true, ind_test)

        # subsample training indices
        ind_train_all, _ = train_test_split(
            ind_dvlp,
            shuffle=False,
            random_state=rng_split,
            test_size=(1 - (par_dtst["train_size"] / dvlp_size)),
        )

        for i_m, par_mdl in enumerate(grid_model):

            # global combo index — must match HPC indexing for seed consistency
            global_i_ds = DATASET_IDS[i_ds]
            combo_idx = global_i_ds * n_dt * n_m + i_dt * n_m + i_m
            model_seed = BASE_SEED + combo_idx

            c = par_mdl["c"]
            K1 = par_mdl["K1"]
            K2 = par_mdl["K2"]
            r = par_mdl["r"]
            lambda_d = par_mdl["lambda_d"]
            lambda_t = par_mdl["lambda_t"]
            alpha = par_mdl["alpha"]
            beta = par_mdl["beta"]
            theta = par_mdl["theta"]
            max_iter = par_mdl["max_iter"]

            try:
                # ---- full train fit → test score ----
                Y_train_full = get_submatrix(Y, ind_train_all)
                Y_train_full[Y_train_full == -1] = 0.0

                model = NRLMF(
                    cfix=c,
                    K1=K1,
                    K2=K2,
                    num_factors=r,
                    lambda_d=lambda_d,
                    lambda_t=lambda_t,
                    alpha=alpha,
                    beta=beta,
                    theta=theta,
                    max_iter=max_iter,
                )
                model.fix_model(
                    np.ones((Y_train_full.shape[0], Y_train_full.shape[1])),
                    Y_train_full.toarray(),
                    simD,
                    simT,
                )
                est_A, est_B = model.U, model.V

                # low-rank prediction trick
                SA = simD @ est_A  # (n, r)
                STB = simT @ est_B  # (m, r)
                prob_full = expit(SA @ STB.T)
                prob_test = get_submatrix(prob_full, ind_test)
                scores_test = get_metrics((Y_test.data + 1) / 2, prob_test.data)

                # ---- repeated holdout CV ----
                splt = ShuffleSplit(
                    n_splits=par_dtst["n_splits"],
                    test_size=par_dtst["val_size"],
                    random_state=rng_split,
                )
                for cv, (ind_train, ind_valid) in enumerate(splt.split(ind_train_all)):
                    ind_train_cv = ind_train_all[ind_train]
                    ind_valid_cv = ind_train_all[ind_valid]

                    Y_train = get_submatrix(Y, ind_train_cv)
                    Y_valid = get_submatrix(Y, ind_valid_cv)
                    Y_train[Y_train == -1] = 0.0

                    model_cv = NRLMF(
                        cfix=c,
                        K1=K1,
                        K2=K2,
                        num_factors=r,
                        lambda_d=lambda_d,
                        lambda_t=lambda_t,
                        alpha=alpha,
                        beta=beta,
                        theta=theta,
                        max_iter=max_iter,
                    )
                    model_cv.fix_model(
                        np.ones((Y_train.shape[0], Y_train.shape[1])),
                        Y_train.toarray(),
                        simD,
                        simT,
                    )
                    est_A_cv, est_B_cv = model_cv.U, model_cv.V

                    SA_cv = simD @ est_A_cv
                    STB_cv = simT @ est_B_cv
                    prob_full_cv = expit(SA_cv @ STB_cv.T)
                    prob_valid = get_submatrix(prob_full_cv, ind_valid_cv)
                    scores_valid = get_metrics((Y_valid.data + 1) / 2, prob_valid.data)

                    dataset_results.append(
                        {
                            # dataset identity
                            "feature_id": ds["feature_id"],
                            "n_features": ds["n_features"],
                            "repeat_id": ds["repeat_id"],
                            # experimental condition
                            "train_size": par_dtst["train_size"],
                            "n_splits": par_dtst["n_splits"],
                            "val_size": par_dtst["val_size"],
                            # hyperparameters
                            "c": c,
                            "K1": K1,
                            "K2": K2,
                            "r": r,
                            "lambda_d": lambda_d,
                            "lambda_t": lambda_t,
                            "alpha": alpha,
                            "beta": beta,
                            "theta": theta,
                            "max_iter": max_iter,
                            # CV fold
                            "cv": int(cv),
                            # validation scores
                            "val_score": scores_valid,
                            # test scores
                            "test_score": scores_test,
                        }
                    )

            except Exception as e:
                print(
                    f"  ERROR at feature_id={ds['feature_id']}, repeat_id={ds['repeat_id']}, "
                    f"train_size={par_dtst['train_size']:.2f}, c={c}: {e}"
                )

    # ---- incremental save after each dataset ----
    all_results.extend(dataset_results)
    with gzip.open(outfile, "wb+", 4) as fout:
        pickle.dump(all_results, fout)
    print(f"  Saved {len(all_results)} rows so far → {outfile}")

print(f"\nDone. Total rows: {len(all_results)}")

Datasets:   0%|          | 0/10 [00:00<?, ?it/s]/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


  Building sim matrices for feature_id=0, repeat_id=0


Datasets:  10%|█         | 1/10 [07:28<1:07:15, 448.38s/it]

  Saved 90 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_nrlmf_local_nb0.gz
  Building sim matrices for feature_id=0, repeat_id=1


/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
Datasets:  20%|██        | 2/10 [14:55<59:41, 447.66s/it]  

...

  Saved 720 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_nrlmf_local_nb0.gz
  Building sim matrices for feature_id=0, repeat_id=8


/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
Datasets:  90%|█████████ | 9/10 [1:27:08<09:58, 598.04s/it]

  Saved 810 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_nrlmf_local_nb0.gz
  Building sim matrices for feature_id=0, repeat_id=9


/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
Datasets: 100%|██████████| 10/10 [1:36:48<00:00, 580.83s/it]

  Saved 900 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_nrlmf_local_nb0.gz

Done. Total rows: 900
